In [0]:
from datetime import datetime
from pyspark.sql import SparkSession ,Row
from delta.tables import DeltaTable

CONTROL_TABLE_PATH = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_control/ingestion_log"


In [0]:

def _ensure_control_table_exists(spark: SparkSession) -> None:

    try:
        spark.read.format("delta").load(CONTROL_TABLE_PATH)
    except Exception:
        empty_df = spark.createDataFrame([], schema="""
            batch_id STRING,
            source_name STRING,
            partition_key STRING,
            status STRING,
            rows_written LONG,
            started_at TIMESTAMP,
            completed_at TIMESTAMP,
            error_message STRING
        """)
        empty_df.write.format("delta").mode("overwrite").save(CONTROL_TABLE_PATH)


In [0]:
def log_ingestion_event(
    spark: SparkSession,
    batch_id: str,
    source_name: str,
    partition_key: str,
    status: str,
    rows_written: int = 0,
    started_at: datetime = None,
    error_message: str = None,
) -> None:

    _ensure_control_table_exists(spark)

    schema = """
        batch_id STRING,
        source_name STRING,
        partition_key STRING,
        status STRING,
        rows_written LONG,
        started_at TIMESTAMP,
        completed_at TIMESTAMP,
        error_message STRING
    """

    row = Row(
        batch_id=batch_id,
        source_name=source_name,
        partition_key=partition_key,
        status=status,
        rows_written=rows_written,
        started_at=started_at or datetime.utcnow(),
        completed_at=datetime.utcnow(),
        error_message=error_message,
    )

    spark.createDataFrame([row], schema=schema).write.format("delta").mode("append").save(CONTROL_TABLE_PATH)

In [0]:
WATERMARK_TABLE_PATH = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_control/watermark_log"
WATERMARK_SCHEMA = "source_name STRING, watermark_value STRING, updated_at TIMESTAMP"


def _ensure_watermark_table_exists(spark: SparkSession) -> None:

    path_exists = False
    

    try:
        dbutils.fs.ls(WATERMARK_TABLE_PATH)
        path_exists = True
    except Exception:
        path_exists = False
        

    if not path_exists or not DeltaTable.isDeltaTable(spark, WATERMARK_TABLE_PATH):
        print(f"[control_table] Watermark table not found at {WATERMARK_TABLE_PATH}. Initializing...")
        empty_df = spark.createDataFrame([], schema=WATERMARK_SCHEMA)
        empty_df.write.format("delta").mode("overwrite").save(WATERMARK_TABLE_PATH)
        print(f"[control_table] Successfully initialized watermark table.")


In [0]:
def get_watermark(spark: SparkSession, source_name: str, default_value: str) -> str:

    _ensure_watermark_table_exists(spark)
    
    df = spark.read.format("delta").load(WATERMARK_TABLE_PATH)
    match = df.filter(df.source_name == source_name).collect()
    
    if match:
        return match[0]["watermark_value"]
    else:
        print(f"[control_table] No entry for '{source_name}'. Using default: {default_value}")
        return default_value



In [0]:
def update_watermark(spark: SparkSession, source_name: str, new_value: str) -> None:

    _ensure_watermark_table_exists(spark)
    delta_table = DeltaTable.forPath(spark, WATERMARK_TABLE_PATH)
    updates_df = spark.createDataFrame(
        [(source_name, new_value, datetime.utcnow())], schema=WATERMARK_SCHEMA
    )
    (
        delta_table.alias("t")
        .merge(updates_df.alias("s"), "t.source_name = s.source_name")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )